In [ ]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/ARR Data/Human Validation/humanData.xlsx" .

In [ ]:
!pip install hazm bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.3 MB/s eta 0:00:00
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9371 sha256=79d3a235389b90c8594a6ad53e22f95161dbcc7e3c70396f7891adf565549c68
  Stored in directory: /root/.cache/pip/wheels/be/12/c9/228313ff5cb722777830302f1d4136de4129932e6a0354c056
Successfully built flashtext


In [ ]:
!pip install -q "transformers<5" "tokenizers<0.22" "bert-score==0.3.12"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 635.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import numpy as np
import pandas as pd
import ast
from hazm import WordTokenizer
from bert_score import score as bert_score
import torch

In [ ]:
df = pd.read_excel("humanData.xlsx")

In [ ]:
ref_col = 'Hop_5_Rationale_Target_Span (Gold)'
pred_col = 'Hop_5_Rationale_Target_Span (Gemini)'

In [ ]:
df = df[df["Hop_1_Triage (Gold)"] != "بی‌ضرر"]
df[pred_col] = df[pred_col].fillna('[]')

In [ ]:
df[ref_col] = df[ref_col].apply(ast.literal_eval)
df[pred_col] = df[pred_col].apply(ast.literal_eval)

In [ ]:
tokenizer = WordTokenizer()

def tokenize(text):
    return tokenizer.tokenize(text)

In [ ]:
def jaccard_similarity(a, b):
    A = set(tokenize(a))
    B = set(tokenize(b))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def max_pairwise_jaccard(pred_spans, gold_spans):
    if not pred_spans or not gold_spans:
        return 0.0

    return max(
        jaccard_similarity(p, g)
        for p in pred_spans
        for g in gold_spans
    )

In [ ]:
def max_pairwise_bertscore(
    pred_spans,
    gold_spans,
    model_type="HooshvareLab/bert-base-parsbert-uncased",
    device=None
):
    if not pred_spans or not gold_spans:
        return 0.0

    max_score = 0.0

    for p in pred_spans:
        P, R, F1 = bert_score(
            [p] * len(gold_spans),
            gold_spans,
            model_type=model_type,
            num_layers=12,
            lang='fa',
            verbose=False,
            device=device,
        )
        max_score = max(max_score, F1.max().item())

    return max_score

In [ ]:
def evaluate_rationales(df, gold_col, pred_col):
    jaccards = []
    bertscores = []

    for index, row in df.iterrows():
        max_pairwise_jaccard_val = max_pairwise_jaccard(row[pred_col], row[gold_col])
        jaccards.append(max_pairwise_jaccard_val)

        bertScore = max_pairwise_bertscore(row[pred_col], row[gold_col])
        bertscores.append(bertScore)

    return {
        "MaxPairwiseJaccard": sum(jaccards) / len(jaccards),
        "MaxPairwiseBERTScore": sum(bertscores) / len(bertscores)
    }

In [ ]:
results = evaluate_rationales(df, gold_col=ref_col, pred_col=pred_col)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

In [ ]:
results

{'MaxPairwiseJaccard': 0.9184210526315789,
 'MaxPairwiseBERTScore': 0.9331726078924379}

In [ ]:
def jaccard(a, b):
    A, B = set(tokenize(a)), set(tokenize(b))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

In [ ]:
def normal_jaccard(pred_spans, gold_spans):
    if not pred_spans or not gold_spans:
        return 0.0

    scores = []
    for p in pred_spans:
        scores.append(max(jaccard(p, g) for g in gold_spans))
    return sum(scores) / len(scores)

In [ ]:
def normal_bertscore(pred_spans, gold_spans, device=None):
    if not pred_spans or not gold_spans:
        return 0.0

    scores = []
    for p in pred_spans:
        _, _, F1 = bert_score(
            [p] * len(gold_spans),
            gold_spans,
            model_type="HooshvareLab/bert-base-parsbert-uncased",
            num_layers=12,
            lang='fa',
            verbose=False,
            device=device,
        )
        scores.append(F1.max().item())
    return sum(scores) / len(scores)

In [ ]:
def evaluate_rationales2(df, gold_col, pred_col):
    jaccards = []
    bertscores = []

    for index, row in df.iterrows():
        jaccard_val = normal_jaccard(row[pred_col], row[gold_col])
        jaccards.append(jaccard_val)

        bertScore = normal_bertscore(row[pred_col], row[gold_col])
        bertscores.append(bertScore)

    return {
        "normalJaccard": sum(jaccards) / len(jaccards),
        "normalBERTScore": sum(bertscores) / len(bertscores)
    }

In [ ]:
results2 = evaluate_rationales2(df, gold_col=ref_col, pred_col=pred_col)

In [ ]:
results2

{'normalJaccard': 0.8654183535762483, 'normalBERTScore': 0.9129392777880032}